In [1]:
import pandas as pd
from pathlib import Path
from src.config import settings
import os

In [4]:
CENSO_PATH      = settings.EXTERNAL_DATA_PATH / 'censo_demografico_2022.csv'
CENSO_PCD_PATH  = settings.EXTERNAL_DATA_PATH / 'censo_demografico_pcd_2022.csv'

In [21]:
# Carregamento das bases
df_censo = pd.read_csv(CENSO_PATH, sep=';')
df_censo_pcd = pd.read_csv(CENSO_PCD_PATH, sep=';')
df_censo_pcd.rename(columns={'Cód.':'cod_ibge', 'Total':'pop_pcd'}, inplace=True)

## CENSO DEMOGRÁFICO

In [23]:
df_censo = df_censo.merge(
    right=df_censo_pcd[['cod_ibge', 'pop_pcd']],
    how='left',
    on='cod_ibge'   
)

In [46]:
RENAME_COLUMNS_CENSO = {
   'Unidade da Federação e Município': 'nome_ente',
   'Total': 'pop_total',
   'Branca': 'pop_branca',
   'Preta': 'pop_preta',
   'Amarela': 'pop_amarela',
   'Parda': 'pop_parda',
   'Indígena': 'pop_indigena'
}

COLS_POP = [
    "pop_branca",
    "pop_preta",
    "pop_amarela",
    "pop_parda",
    "pop_indigena",
    "pop_pcd",
    "pop_pessoas_negras",
]

COLS_NUM = COLS_POP + ["pop_total"]

In [32]:
df_censo = df_censo.rename(columns=RENAME_COLUMNS_CENSO)

In [34]:
# DEFINE COLUNA PESSOAS NEGRAS
df_censo["pop_pessoas_negras"] = df_censo["pop_preta"] + df_censo["pop_parda"]

In [47]:
# CRIA A COLUNA PERCENTUAL PARA CADA CATEGORIA DE POPULAÇÃO


df_censo[COLS_NUM] = df_censo[COLS_NUM].apply(
    pd.to_numeric, errors="coerce"
)

for col in COLS_POP:
    df_censo[f"rel_{col}"] = df_censo[col] / df_censo["pop_total"]

In [48]:
df_censo

,nivel,cod_ibge,nome_ente,pop_total,pop_branca,pop_preta,pop_amarela,pop_parda,pop_indigena,pop_pcd,pop_pessoas_negras,rel_pop_branca,rel_pop_preta,rel_pop_amarela,rel_pop_parda,rel_pop_indigena,rel_pop_pcd,rel_pop_pessoas_negras
0,UF,11,Rondônia,1581196,486123,136793,4257.0,936708,17278.0,108535,1073501,0.307440,0.086512,0.002692,0.592405,0.010927,0.068641,0.678917
1,UF,12,Acre,830018,177992,71086,1878.0,549889,29163.0,58983,620975,0.214444,0.085644,0.002263,0.662502,0.035135,0.071062,0.748146
2,UF,13,Amazonas,3941613,725007,193667,5963.0,2711618,305243.0,266814,2905285,0.183937,0.049134,0.001513,0.687946,0.077441,0.067692,0.737080
3,UF,14,Roraima,636707,131260,49195,784.0,364494,89882.0,34316,413689,0.206154,0.077265,0.001231,0.572467,0.141167,0.053896,0.649732
4,UF,15,Pará,8120131,1570281,793621,12432.0,5673446,69180.0,577865,6467067,0.193381,0.097735,0.001531,0.698689,0.008520,0.071164,0.796424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5592,MU,5222005,Vianópolis (GO),14956,6328,954,14.0,7645,15.0,735,8599,0.423108,0.063787,0.000936,0.511166,0.001003,0.049144,0.574953
5593,MU,5222054,Vicentinópolis (GO),8768,3349,715,10.0,4683,11.0,664,5398,0.381957,0.081547,0.001141,0.534101,0.001255,0.075730,0.615648
5594,MU,5222203,Vila Boa (GO),4215,899,861,3.0,2444,8.0,318,3305,0.213286,0.204270,0.000712,0.579834,0.001898,0.075445,0.784104
5595,MU,5222302,Vila Propício (GO),5815,1407,533,24.0,3844,5.0,411,4377,0.241960,0.091660,0.004127,0.661049,0.000860,0.070679,0.752709
